# Notebook: 00 Preprocess Images
### Purpose: TIFF → PNG and refresh annotation paths.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
# ponytail: PROJECT_ROOT or parent of notebooks/
root = Path(os.getenv("PROJECT_ROOT") or (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)).resolve()
sys.path[:0] = [str(root), str(root / "src")]

from src.data.image_loader import validate_image_directory
from src.utils.config import Config
from src.utils.helpers import init_notebook, p, t
from src.utils.image_converter import ImageConverter

config = Config.load(root=root)
init_notebook(config.train.seed)


#### Validate → convert → update annotations → re-validate


In [ ]:
t("Validating train images")
train_results = validate_image_directory(config.paths.train_images)

t("Converting train images")
converter = ImageConverter(
    source_dir=config.paths.train_images,
    target_dir=config.paths.train_images,
)
stats = converter.convert_batch(overwrite=False)

if config.paths.eval_images and config.paths.eval_images.exists():
    t("Converting eval images")
    eval_stats = ImageConverter(
        source_dir=config.paths.eval_images,
        target_dir=config.paths.eval_images,
    ).convert_batch(overwrite=False)

if config.paths.annotations and Path(config.paths.annotations).exists():
    t("Updating annotations to reference PNG files")
    converter.update_annotations(
        annotations_path=config.paths.annotations,
        create_backup=True,
    )

t("Validating converted images")
final_results = validate_image_directory(config.paths.train_images)
t("Conversion Summary")
p("train validate", train_results)
p("train convert", stats)
p("final validate", final_results)
if config.paths.eval_images and config.paths.eval_images.exists():
    p("eval validate", validate_image_directory(config.paths.eval_images))
